## Survival Modeling with Cox Proportional Hazards

### Objective
This notebook builds a Cox Proportional Hazards (Cox PH) model using the engineered METABRIC clinical dataset.

Main goals:
- fit a Cox PH model
- interpret coefficients and hazard ratios
- evaluate concordance index
- inspect statistically important variables

---

## 목적
이 노트북의 목적은 feature engineering이 완료된 METABRIC 임상 데이터를 사용하여 Cox Proportional Hazards 모델을 학습하는 것이다.

핵심 목표:
- Cox PH 모델 학습
- 계수 및 hazard ratio 해석
- concordance index 평가
- 유의한 변수 확인

In [16]:
# 1. Import
import pandas as pd
import numpy as np

from lifelines import CoxPHFitter

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [17]:
# 2. Load processed dataset
file_path = "../data/processed/metabric_clinical_featurized.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

Shape: (1353, 12)


,Age at Diagnosis,Lymph nodes examined positive,Tumor Size,Neoplasm Histologic Grade_2.0,Neoplasm Histologic Grade_3.0,Tumor Stage_2.0,Tumor Stage_3.0,Tumor Stage_4.0,ER Status_Positive,HER2 Status_Positive,time,event
0,75.6500,10.0000,22.0000,0,1,1,0,0,1,0,140.5000,0
1,43.1900,0.0000,10.0000,0,1,0,0,0,1,0,84.6333,0
2,48.8700,1.0000,15.0000,1,0,1,0,0,1,0,163.7000,1
3,47.6800,3.0000,25.0000,1,0,1,0,0,1,0,164.9333,0
4,76.9700,8.0000,40.0000,0,1,1,0,0,1,0,41.3667,1


In [18]:
# 3. Basic validation
print(df.info())

print("Missing values:", df.isnull().sum().sum())
print("Event distribution:\n", df["event"].value_counts())
print("\nTime summary:\n", df["time"].describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1353 entries, 0 to 1352
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age at Diagnosis               1353 non-null   float64
 1   Lymph nodes examined positive  1353 non-null   float64
 2   Tumor Size                     1353 non-null   float64
 3   Neoplasm Histologic Grade_2.0  1353 non-null   int64  
 4   Neoplasm Histologic Grade_3.0  1353 non-null   int64  
 5   Tumor Stage_2.0                1353 non-null   int64  
 6   Tumor Stage_3.0                1353 non-null   int64  
 7   Tumor Stage_4.0                1353 non-null   int64  
 8   ER Status_Positive             1353 non-null   int64  
 9   HER2 Status_Positive           1353 non-null   int64  
 10  time                           1353 non-null   float64
 11  event                          1353 non-null   int64  
dtypes: float64(4), int64(8)
memory usage: 127.0 KB
N

## Modeling Inputs

- `time` : survival duration
- `event` : event indicator (1 = death, 0 = censored/alive)
- all remaining columns : engineered clinical features

---

## 모델 입력 구조

- `time` : 생존 기간
- `event` : 사건 발생 여부 (1 = 사망, 0 = 검열/생존)
- 나머지 컬럼 : feature engineering 완료된 임상 변수

In [19]:
# 4. Fix Con PH model
cph = CoxPHFitter()

cph.fit(
    df,
    duration_col="time",
    event_col="event"
)

print("Model fitting completed.")

Model fitting completed.


In [20]:
# 5. Model summary
cph.summary

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
Age at Diagnosis,0.0054,1.0055,0.0040,-0.0024,0.0132,0.9976,1.0133,0.0000,1.3677,0.1714,2.5446
Lymph nodes examined positive,0.0577,1.0594,0.0108,0.0366,0.0788,1.0373,1.0820,0.0000,5.3596,0.0000,23.5151
Tumor Size,0.0088,1.0088,0.0027,0.0035,0.0141,1.0035,1.0142,0.0000,3.2287,0.0012,9.6513
Neoplasm Histologic Grade_2.0,0.3644,1.4396,0.2393,-0.1047,0.8334,0.9006,2.3012,0.0000,1.5225,0.1279,2.9672
Neoplasm Histologic Grade_3.0,0.5939,1.8110,0.2381,0.1273,1.0605,1.1357,2.8878,0.0000,2.4946,0.0126,6.3091
Tumor Stage_2.0,0.3981,1.4890,0.1201,0.1627,0.6335,1.1767,1.8842,0.0000,3.3146,0.0009,10.0897
Tumor Stage_3.0,0.6132,1.8463,0.2098,0.2019,1.0244,1.2237,2.7854,0.0000,2.9223,0.0035,8.1687
Tumor Stage_4.0,1.1942,3.3011,0.4187,0.3735,2.0150,1.4529,7.5005,0.0000,2.8520,0.0043,7.8466
ER Status_Positive,-0.2523,0.7770,0.1185,-0.4845,-0.0201,0.6160,0.9801,0.0000,-2.1292,0.0332,4.9111


In [21]:
# 6. Core metrics
print("Concordance Index:", round(cph.concordance_index_, 4))
print("Log-likelihood:", round(cph.log_likelihood_, 4))

Concordance Index: 0.702
Log-likelihood: -2982.0901


In [22]:
# 7. Hazard ratio table
hr_table = cph.summary[[
    "coef",
    "exp(coef)",
    "exp(coef) lower 95%",
    "exp(coef) upper 95%",
    "se(coef)",
    "z",
    "p",
    "-log2(p)"
]].sort_values(by="p")

hr_table

,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
covariate,,,,,,,,
Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0108,5.3596,0.0000,23.5151
HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.1299,3.5046,0.0005,11.0948
Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.1201,3.3146,0.0009,10.0897
Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0027,3.2287,0.0012,9.6513
Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.2098,2.9223,0.0035,8.1687
Tumor Stage_4.0,1.1942,3.3011,1.4529,7.5005,0.4187,2.8520,0.0043,7.8466
Neoplasm Histologic Grade_3.0,0.5939,1.8110,1.1357,2.8878,0.2381,2.4946,0.0126,6.3091
ER Status_Positive,-0.2523,0.7770,0.6160,0.9801,0.1185,-2.1292,0.0332,4.9111
Neoplasm Histologic Grade_2.0,0.3644,1.4396,0.9006,2.3012,0.2393,1.5225,0.1279,2.9672


In [23]:
# 8. Significant variables only
significant_features = hr_table[hr_table["p"] < 0.05].copy()
significant_features

,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
covariate,,,,,,,,
Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0108,5.3596,0.0000,23.5151
HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.1299,3.5046,0.0005,11.0948
Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.1201,3.3146,0.0009,10.0897
Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0027,3.2287,0.0012,9.6513
Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.2098,2.9223,0.0035,8.1687
Tumor Stage_4.0,1.1942,3.3011,1.4529,7.5005,0.4187,2.8520,0.0043,7.8466
Neoplasm Histologic Grade_3.0,0.5939,1.8110,1.1357,2.8878,0.2381,2.4946,0.0126,6.3091
ER Status_Positive,-0.2523,0.7770,0.6160,0.9801,0.1185,-2.1292,0.0332,4.9111


## Interpretation Guide

- `coef > 0` : higher risk (hazard increases)
- `coef < 0` : protective effect (hazard decreases)
- `exp(coef) > 1` : increased hazard
- `exp(coef) < 1` : decreased hazard

For example:
- hazard ratio = 1.20 → 20% higher hazard
- hazard ratio = 0.80 → 20% lower hazard

---

## Hazard Ratio 해석 기준

- `coef > 0` : 위험 증가
- `coef < 0` : 보호 효과
- `exp(coef) > 1` : hazard 증가
- `exp(coef) < 1` : hazard 감소

예시:
- hazard ratio = 1.20 → 위험 20% 증가
- hazard ratio = 0.80 → 위험 20% 감소

In [24]:
# 9. Top risk-increasing features
top_risk_features = hr_table.sort_values(by="exp(coef)", ascending=False).head(10)
top_risk_features

,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
covariate,,,,,,,,
Tumor Stage_4.0,1.1942,3.3011,1.4529,7.5005,0.4187,2.8520,0.0043,7.8466
Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.2098,2.9223,0.0035,8.1687
Neoplasm Histologic Grade_3.0,0.5939,1.8110,1.1357,2.8878,0.2381,2.4946,0.0126,6.3091
HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.1299,3.5046,0.0005,11.0948
Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.1201,3.3146,0.0009,10.0897
Neoplasm Histologic Grade_2.0,0.3644,1.4396,0.9006,2.3012,0.2393,1.5225,0.1279,2.9672
Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0108,5.3596,0.0000,23.5151
Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0027,3.2287,0.0012,9.6513
Age at Diagnosis,0.0054,1.0055,0.9976,1.0133,0.0040,1.3677,0.1714,2.5446


In [25]:
# 10. Top protective features
top_protective_features = hr_table.sort_values(by="exp(coef)", ascending=True).head(10)
top_protective_features

,coef,exp(coef),exp(coef) lower 95%,exp(coef) upper 95%,se(coef),z,p,-log2(p)
covariate,,,,,,,,
ER Status_Positive,-0.2523,0.7770,0.6160,0.9801,0.1185,-2.1292,0.0332,4.9111
Age at Diagnosis,0.0054,1.0055,0.9976,1.0133,0.0040,1.3677,0.1714,2.5446
Tumor Size,0.0088,1.0088,1.0035,1.0142,0.0027,3.2287,0.0012,9.6513
Lymph nodes examined positive,0.0577,1.0594,1.0373,1.0820,0.0108,5.3596,0.0000,23.5151
Neoplasm Histologic Grade_2.0,0.3644,1.4396,0.9006,2.3012,0.2393,1.5225,0.1279,2.9672
Tumor Stage_2.0,0.3981,1.4890,1.1767,1.8842,0.1201,3.3146,0.0009,10.0897
HER2 Status_Positive,0.4552,1.5765,1.2222,2.0336,0.1299,3.5046,0.0005,11.0948
Neoplasm Histologic Grade_3.0,0.5939,1.8110,1.1357,2.8878,0.2381,2.4946,0.0126,6.3091
Tumor Stage_3.0,0.6132,1.8463,1.2237,2.7854,0.2098,2.9223,0.0035,8.1687


> Note: Hazard ratios from very small subgroups (e.g., rare tumor stages) should be interpreted cautiously.
>
> 주의: 표본 수가 매우 작은 subgroup에서 추정된 hazard ratio는 해석에 주의가 필요하다.

In [26]:
# 11. Coefficient ranking table
coef_table = cph.summary[["coef", "exp(coef)", "p"]].sort_values(by="coef", ascending=False)
coef_table

,coef,exp(coef),p
covariate,,,
Tumor Stage_4.0,1.1942,3.3011,0.0043
Tumor Stage_3.0,0.6132,1.8463,0.0035
Neoplasm Histologic Grade_3.0,0.5939,1.8110,0.0126
HER2 Status_Positive,0.4552,1.5765,0.0005
Tumor Stage_2.0,0.3981,1.4890,0.0009
Neoplasm Histologic Grade_2.0,0.3644,1.4396,0.1279
Lymph nodes examined positive,0.0577,1.0594,0.0000
Tumor Size,0.0088,1.0088,0.0012
Age at Diagnosis,0.0054,1.0055,0.1714


In [27]:
# 12. Optional: proportional hazards assumption check
cph.check_assumptions(df, p_value_threshold=0.05, show_plots=False)

The ``p_value_threshold`` is set at 0.05. Even under the null hypothesis of no violations, some
covariates will be below the threshold by chance. This is compounded when there are many covariates.
Similarly, when there are lots of observations, even minor deviances from the proportional hazard
assumption will be flagged.

With that in mind, it's best to use a combination of statistical tests and visual tests to determine
the most serious violations. Produce visual plots using ``check_assumptions(..., show_plots=True)``
and looking for non-constant lines. See link [A] below for a full example.





1. Variable 'Age at Diagnosis' failed the non-proportional test: p-value is 0.0402.

   Advice 1: the functional form of the variable 'Age at Diagnosis' might be incorrect. That is,
there may be non-linear terms missing. The proportional hazard test used is very sensitive to
incorrect functional forms. See documentation in link [D] below on how to specify a functional form.

   Advice 2: try binning the variable 'Age at Diagnosis' using pd.cut, and then specify it in
`strata=['Age at Diagnosis', ...]` in the call in `.fit`. See documentation in link [B] below.

   Advice 3: try adding an interaction term with your time variable. See documentation in link [C]
below.


2. Variable 'Tumor Stage_2.0' failed the non-proportional test: p-value is 0.0320.

   Advice: with so few unique values (only 2), you can include `strata=['Tumor Stage_2.0', ...]` in
the call in `.fit`. See documentation in link [E] below.

3. Variable 'Tumor Stage_3.0' failed the non-proportional test: p-value is 0.001

[]

In [28]:
# 13. Save model summary
summary_path = "../data/processed/coxph_model_summary.csv"
cph.summary.to_csv(summary_path)

print(f"Saved model summary to: {summary_path}")

Saved model summary to: ../data/processed/coxph_model_summary.csv


In [29]:
# 14. Save HR table
hr_path = "../data/processed/coxph_hazard_ratios.csv"
hr_table.to_csv(hr_path)

print(f"Saved hazard ratio table to: {hr_path}")

Saved hazard ratio table to: ../data/processed/coxph_hazard_ratios.csv


## Modeling Summary

The Cox Proportional Hazards model was successfully fitted on the engineered METABRIC clinical dataset.

Key outputs:
- fitted regression coefficients
- hazard ratios
- p-values for statistical significance
- concordance index for ranking performance

This model provides an interpretable baseline for identifying clinical variables associated with survival risk.

---

## 모델링 요약

Cox Proportional Hazards 모델이 feature engineering 완료된 METABRIC 데이터에 성공적으로 학습되었다.

핵심 결과:
- 회귀계수(coefficient)
- hazard ratio
- 통계적 유의성(p-value)
- concordance index

이 모델은 생존 위험과 관련된 임상 변수를 해석 가능한 방식으로 확인할 수 있는 baseline 모델이다.

---

## Practical Notes

If some variables show unstable coefficients or very large hazard ratios,
this may be caused by:
- rare categories
- small subgroup sizes
- collinearity
- violation of proportional hazards assumption

These issues should be reviewed during model refinement.

---

## 실무적 해석 메모

일부 변수에서 계수가 불안정하거나 hazard ratio가 매우 크게 나타난다면, 다음 원인 가능성을 검토해야 한다.

- 희귀 범주
- 작은 subgroup 크기
- 다중공선성
- proportional hazards 가정 위반

이러한 부분은 이후 모델 리팩토링 단계에서 점검해야 한다.